In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from src.data import load_csv_df
from src.plotting import set_plotting_style

In [ ]:
# Already prepared accent palette applying using function from mine src module (plotting)
set_plotting_style()

In [ ]:
DF_PATH = "../data/raw/raw-data.csv"

# Safely loading dataframe using function from mine src module (data)
df = load_csv_df(
    df_path=DF_PATH,
)

In [ ]:
df = df.set_index("Country name")
display(df.head(5)) # Index replaces with "Country name" identifier

In [ ]:
num_col = df.select_dtypes(include="number").columns
df = df[num_col].fillna(df[num_col].median())
print(f"Missing: {df.isna().sum().sum()} values")

In [ ]:
duplicated_cols = ["Total in km2", "% of world population"]
df_prep = df.drop(columns=duplicated_cols)
print(df_prep.columns)

In [ ]:
num_col = df_prep.select_dtypes(include="number").columns
skewed_cols = [col for col in num_col if col != "HDI"]
df_prep[skewed_cols] = np.log1p(df_prep[skewed_cols])

df_skewness = df_prep[skewed_cols].skew()
print(df_skewness.sort_values(ascending=False)) # Columns successfully log-transformed

In [ ]:
df_prep["Land per capita"] = df_prep["Land in km2"] / df_prep["Population"]
df_prep["Dev demographic index"] = df_prep["HDI"] / df_prep["Fertility rate"]
"""Feature engineering:
----------------------------
1. "Land per capita" (Land in km2 / Population):
    - Measure how much physical land area exists each person in a country.
    - Helps model clearly group huge, low density country and seperate them.
2. "Dev demographic index" (HDI / Fertility rate):
    - Combines a country's quality of life and birth rate into a simple score.
    - Rich, developed countries with low birth rates get a high score,
    while developing countries with high birth rates get a low score.
"""

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_prep)

X_scaled_df = pd.DataFrame(
    X_scaled,
    index=df_prep.index,
    columns=df_prep.columns,
)

print(f"Processed dataframe Shape: {X_scaled_df.shape}")
display(X_scaled_df.head(5)) # Data successfully scaled

Processed dataframe Shape: (195, 9)


,Land in km2,Inland water in km2,% water,Population,Population density,HDI,Fertility rate,Land per capita,Dev demographic index
Country name,,,,,,,,,
Russia,1.262288,1.072830,1.013228,1.210663,-1.801045,0.617095,-0.845314,0.958271,0.727524
Canada,1.157692,1.088406,1.532612,0.780616,-2.585632,1.338322,-1.072927,1.069353,1.407167
China,1.162262,0.997449,0.712391,1.918684,0.479567,0.381180,-1.640508,0.468253,1.284983
United States,1.158764,1.023635,0.977621,1.485679,-0.502857,1.331581,-0.608133,0.686716,0.995708
Brazil,1.144614,0.863054,-0.392106,1.336013,-0.791998,0.307035,-0.665342,0.748928,0.398306
